# Lecture 04 solutions — Imbalanced learning

Complete executable worked answers.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    auc,
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    brier_score_loss,
)
from xgboost import XGBClassifier


def repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "data" / "fraud.csv").exists():
            return path
    raise FileNotFoundError("Could not find data/fraud.csv")


ROOT = repo_root()
data = pd.read_csv(ROOT / "data" / "fraud.csv")
features = [
    "amount",
    "hour",
    "card_present",
    "international",
    "account_age_days",
    "transactions_24h",
    "merchant_risk",
]
X, y = data[features], data["fraud"].astype(int)
print(f"rows={len(data)}, positives={y.sum()}, prevalence={y.mean():.3%}")
# Split first. The test rows remain untouched until all choices are frozen.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=14, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=15, stratify=y_dev
)
print("sizes:", len(X_train), len(X_valid), len(X_test))


def empirical_cost(actual, scores, threshold, c_fn=20.0, c_fp=1.0):
    predicted = scores >= threshold
    false_negative = np.sum((actual.to_numpy() == 1) & ~predicted)
    false_positive = np.sum((actual.to_numpy() == 0) & predicted)
    return c_fn * false_negative + c_fp * false_positive

## 1. Baseline

Always-negative accuracy is $1-\pi$ and recall is zero; accuracy reflects majority prevalence.

In [ ]:
negative, positive = np.bincount(y_test, minlength=2)
print(
    {
        "negative": negative,
        "positive": positive,
        "prevalence": positive / len(y_test),
        "always-negative accuracy": negative / len(y_test),
        "recall": 0.0,
    }
)

## 2. Posterior shift

Dividing $q$ by $1-q$ gives weighted odds. At $q=1/2$, $p=w_0/(w_0+w_1)$, below one-half if positives weigh more.

## 2a. Why class weights shift the posterior

At fixed features $x$, let the natural positive probability be $p$. The conditional weighted Bernoulli risk for a proposed probability $q$ is $R(q|x)=-w_1p\log q-w_0(1-p)\log(1-q)$. Its derivative is $-w_1p/q+w_0(1-p)/(1-q)$. Setting this to zero gives $q/(1-q)=(w_1/w_0)p/(1-p)$. Thus weighting shifts odds even with an ideal learner; fitted weighted probabilities should not automatically be read as natural risk.

In [ ]:
p = np.linspace(0.001, 0.2, 100)
w0, w1 = 1.0, 10.0
q = w1 * p / (w1 * p + w0 * (1 - p))
plt.plot(p, q)
plt.axhline(0.5, color="black", linestyle="--")
plt.axvline(w0 / (w0 + w1), color="black", linestyle=":")
plt.xlabel("natural p")
plt.ylabel("weighted q")

## 3. Model comparison

AP uses recall increments while trapezoid interpolates curve segments. ROC-AUC is a ranking measure, not calibration or a selected operating point.

In [ ]:
negative, positive = np.bincount(y_train, minlength=2)
ratio = negative / positive
models = {
    "Logistic unweighted": make_pipeline(
        StandardScaler(), LogisticRegression(C=1, max_iter=2000, random_state=14)
    ),
    "Logistic balanced": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1, class_weight="balanced", max_iter=2000, random_state=14
        ),
    ),
    "XGBoost weighted": XGBClassifier(
        n_estimators=180,
        max_depth=3,
        learning_rate=0.06,
        min_child_weight=4,
        subsample=0.85,
        colsample_bytree=0.9,
        reg_lambda=2,
        scale_pos_weight=ratio,
        eval_metric="logloss",
        n_jobs=1,
        random_state=14,
    ),
}
for model in models.values():
    model.fit(X_train, y_train)
valid_scores = {
    name: model.predict_proba(X_valid)[:, 1] for name, model in models.items()
}
rows = []
for name, scores in valid_scores.items():
    precision, recall, _ = precision_recall_curve(y_valid, scores)
    rows.append(
        {
            "model": name,
            "AP": average_precision_score(y_valid, scores),
            "trapezoid_PR_area": auc(recall[::-1], precision[::-1]),
            "ROC_AUC": roc_auc_score(y_valid, scores),
            "Brier": brier_score_loss(y_valid, scores),
        }
    )
pd.DataFrame(rows).set_index("model").round(4)

## 4. Validation thresholds and one test report

In [ ]:
def empirical_cost(actual, scores, threshold, c_fn=20.0, c_fp=1.0):
    predicted = scores >= threshold
    fn = np.sum((actual.to_numpy() == 1) & ~predicted)
    fp = np.sum((actual.to_numpy() == 0) & predicted)
    return c_fn * fn + c_fp * fp


C_FN, C_FP = 20.0, 1.0
grid = np.unique(np.r_[0.0, np.linspace(0.01, 0.99, 197), 1.0])
thresholds = {}
for name, scores in valid_scores.items():
    costs = [empirical_cost(y_valid, scores, t, C_FN, C_FP) for t in grid]
    thresholds[name] = float(grid[int(np.argmin(costs))])
pd.DataFrame(
    {
        "threshold": thresholds,
        "validation_cost": {
            n: empirical_cost(y_valid, valid_scores[n], t, C_FN, C_FP)
            for n, t in thresholds.items()
        },
        "alert_rate": {n: np.mean(valid_scores[n] >= t) for n, t in thresholds.items()},
    }
).T

In [ ]:
rows = []
for name, model in models.items():
    scores = model.predict_proba(X_test)[:, 1]
    prediction = scores >= thresholds[name]
    tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
    rows.append(
        {
            "model": name,
            "threshold": thresholds[name],
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "precision": precision_score(y_test, prediction, zero_division=0),
            "recall": recall_score(y_test, prediction, zero_division=0),
            "AP": average_precision_score(y_test, scores),
            "ROC_AUC": roc_auc_score(y_test, scores),
            "cost": C_FN * fn + C_FP * fp,
            "test_alert_rate": prediction.mean(),
        }
    )
pd.DataFrame(rows).set_index("model").round(4)

## Interpretation

Thresholds depend on cost assumptions and validation data. Weighted probabilities may shift. The synthetic results do not justify real decisions.